<img src="./images/logo-UVAIA-original.png" width= 200px>   

# Agentes IA con LangChain     

En este notebook se muestra paso a paso como crear un agente sencillo, para el que se implementarán varias *tools* simples que le añadirán funcionalidades enfocadas a resolver tareas específicas.   
Como "motor" de este agente se usará el modelo de lenguaje largo (LLM) QWEN-3 en una de las versiones disponible dentro de **Hugging Face**.   

La implementación se realiza paso a paso, para que resulte comprensible el proceso de implementación de este tipo de herramientas de IA.   

¡Nos ponemos manos a la obra!

## 1.- Instalación de librerías.   

El primer paso a ejecutar es la instalación de las librerías necesarias para poder trabajar en la implementación de nuestro agente.     

Las librerías a instalar son:   
- langchain-openai,
- langchain-huggingface,
- langchain-comunity,
- duckduckgo-search,
- langgraph,
- langchain,
- ddgs

## 2.- Importar librerías

A continuación, importamos los elementos necesarios.

In [1]:
import warnings
import ipywidgets as widgets


from IPython.display import display, Code
from langchain.agents import create_agent

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from sympy import sympify


warnings.filterwarnings("ignore")

C:\Users\Hichem\AppData\Local\Temp\ipykernel_5760\2688460436.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


## 3.- Definir e implementar nuestra primera `TOOL`  
En este bloque de código crearemos la primera herramienta (`Tool`). En este caso, se trata de una función que es capaz de evaluar/realizar cálculos matemáticos usando para ello la función `sympify`de la librería `sympy`, que transforma cadenas de texto en expresiones matemáticas.   

También tenemos definida otra `Tool`para que se realicen búsquedas de información en Internet, en este caso usando el buscador DuckDuckGo, a través de su API.   

Cuando se implementa una `Tool` se deben considerar tres elementos o partes fundamentales:   

1.- El **nombre de la función**, que será el que utilice el agente para identificar y llamar a la `tool` en cuestión. Debemos usar nombres que sean lo suficientemente descriptivos para tener claro que es lo que hace.   
2.- El **docstring**. Esta es, sin duda, el elemento más importante en la creación de una `tool`. El agente va a leer ese texto para decidir si debe usar o no la `tool`en cuestión y como debe ser su *signatura* para invocarla. Si el **docstring** está incompleto o no describe correctamente la utilidad de la `tool`, el agente no tendrá claro cuándo y/o cómo debe usarla.    
3.- La **lógica** de la función implementada: Es el código que se ejecuta cuando el agente decide usar la `tool`. Debe ***devolver siempre un texto (`str`)***, porque el agente necesita leer el resultado para continuar razonando.   

En conclusión, tenemos un esquema como este:    
```python
    @tool
    def nombre_descriptivo(parámetros):    # ← parte 1: nombre
        """
        Qué hace la tool.                  # ← parte 2: docstring
        Args: qué parámetros recibe.             (el agente lee esto)
        Returns: qué devuelve.
        """
        # lógica de la función            # ← parte 3: código
        return "resultado como texto"

```

#### El decorador **@tool**
Un decorador en Python es algo que se coloca encima de una función para añadirle funcionalidad extra sin modificar su código interno.
En este caso, **@tool** transforma una función Python normal en una herramienta que el agente puede usar. Sin él, el agente no sabría que esa función existe ni podría llamarla.       

A continuación, pasamos a implementar el código de las dos primeras `tools` de nuestro agente:

In [2]:
# --- Definir herramientas ---
@tool
def calculadora(expresion: str) -> str:
    """Evalúa expresiones matemáticas de forma segura. Ejemplo: 'sqrt(144)' o '2**10'"""
    try:
        resultado = sympify(expresion)
        return str(resultado)
    except Exception:
        return "Expresión matemática no válida"

search = DuckDuckGoSearchRun()

@tool
def busqueda_web(consulta: str) -> str:
    """Busca información actual en la web sobre cualquier tema."""
    return search.run(consulta)


### Ejercicio: implementación de una herramienta de información sobre países
***Enunciado***   

Se pide implementar una tool llamada **info_pais** que permita a un agente conversacional responder preguntas básicas sobre países.    

La **tool** debe:

- Recibir el nombre de un país y un dato concreto que se quiere consultar
- Devolver la información solicitada
- Soportar las consultas: capital, idioma y continente

Se dispone de los siguientes países: `españa`, `francia`, `alemania`, `italia`, `portugal`, `mexico`, `argentina`, `japon`, `china` y `canada`.   

Una vez implementada la tool, incorporarla al agente que estamos creando y que la use para responder preguntas como:

- *"¿Cuál es la capital de Japón?"*
- *"¿Qué idioma se habla en Brasil?"*
- *"¿En qué continente está Canadá?"*

In [3]:
# Tu código aquí #

In [4]:
import Utilidades as utl
utl.mostrar_boton_solucion(1)

Button(button_style='warning', description='Ver solución', style=ButtonStyle())

Output()

In [5]:

# -----------------------------------------------
# TOOL: info_pais
# -----------------------------------------------
@tool
def info_pais(pais: str, dato: str) -> str:
    """
    Devuelve información básica sobre un país.

    Args:
        pais: Nombre del país en minúsculas (ej: 'españa', 'japon').
        dato: Tipo de dato a consultar: 'capital', 'idioma' o 'continente'.

    Returns:
        El dato solicitado sobre el país.
    """

    paises = {
        "españa":    {"capital": "Madrid",         "idioma": "Español",   "continente": "Europa"},
        "francia":   {"capital": "París",           "idioma": "Francés",   "continente": "Europa"},
        "alemania":  {"capital": "Berlín",          "idioma": "Alemán",    "continente": "Europa"},
        "italia":    {"capital": "Roma",            "idioma": "Italiano",  "continente": "Europa"},
        "portugal":  {"capital": "Lisboa",          "idioma": "Portugués", "continente": "Europa"},
        "mexico":    {"capital": "Ciudad de México","idioma": "Español",   "continente": "América"},
        "argentina": {"capital": "Buenos Aires",    "idioma": "Español",   "continente": "América"},
        "japon":     {"capital": "Tokio",           "idioma": "Japonés",   "continente": "Asia"},
        "china":     {"capital": "Pekín",           "idioma": "Chino mandarín", "continente": "Asia"},
        "canada":    {"capital": "Ottawa",          "idioma": "Inglés y Francés", "continente": "América"},
    }

    pais = pais.lower()
    dato = dato.lower()

    if pais not in paises:
        return f"No tengo información sobre '{pais}'. Países disponibles: {', '.join(paises.keys())}."

    if dato not in {"capital", "idioma", "continente"}:
        return "Dato no válido. Consulta disponibles: 'capital', 'idioma' o 'continente'."

    return f"El {dato} de {pais.capitalize()} es: {paises[pais][dato]}."

En la siguiente línea de código se establece la lista de herramientas para que el agente pueda utilizarla en sus razonamientos y respuestas.

In [6]:
tools = [calculadora, busqueda_web, info_pais]

<!-- Nota o aviso (amarillo) -->
<div style="background-color: #fff3cd; color: #000000 ;padding: 15px; border-radius: 5px;">
<h5>⚠️ Una <b>tool</b> no tiene por qué hacer cálculos: simplemente <i>debe devolver información útil que el agente no podría obtener por sí solo</i> de forma fiable.</h5>
</div>

## 4.- "Conectar" con el LLM que se utilizará como "motor" de nuestro Agente.   

En esta ocasión, a través de la API de HuggingFace, establecemos la conexión con el LLM escogido (`Qwen3-8B`).    

Es necesario disponer (o crear, si no se tiene) una cuenta en HF y gestionar la creación de un API token que, para este caso, solo necesita ser de tipo lectura (READ). Este se debe indicar en la linea indicada en el código.

In [7]:
# --- Inicializar LLM desde HuggingFace ---
llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-8B",       
    task="text-generation",
    max_new_tokens=1024,
    temperature=0.1,
    huggingfacehub_api_token="",  # 👈 Pon aquí tu token de HuggingFace
)

llm = ChatHuggingFace(llm=llm_endpoint)

agent = create_agent(
    model=llm,
    tools=tools,
)

## 5.- Lanzar la consulta.    

En el siguiente bloque, utilizando la función `agent.invoke()` se lanza la consulta a ejecutar por el agente. Esta es una cadena de texto plano con la consulta realizada en lenguaje natural.

In [8]:
# --- Ejecutar ---
response = agent.invoke({
    "messages": [("user", "¿Cuál es la raíz cuadrada de 144 y puedes buscar noticias recientes sobre ese número y puedes decirme cúal es la capital de España?")]
})

# print(response["messages"][-1].content)

In [9]:
# Renderizamos la respuesta del agente, accediendo al último mensaje de la conversación y mostrando su contenido
from IPython.display import Markdown, display # importamos las funciones necesarias para mostrar la respuesta en formato Markdown
display(Markdown(response['messages'][-1].content)) # mostramos la respuesta del agente renderizada



La raíz cuadrada de 144 es **12**.  

En cuanto a noticias recientes sobre el número 144:  
- En WhatsApp, **144** puede significar que alguien está enfermo (1 = "I", 4 = "feel", 4 = "sick").  
- También se mencionó a un tenista que ganó 14 grandes torneos.  
- Hay referencias a noticias globales y eventos actuales.  

La capital de España es **Madrid**.  

¿Necesitas más detalles sobre alguno de estos temas?

----
Hasta aquí la creación de nuestro primer agente. Como se ha comprobado, es relativamente sencillo implementar un agente básico y comprobar su correcto funcionamiento.     

Nuestro paso siguiente es configurar un ***historial de conversaciones***, para poder disponer de toda la lista de mensajes intercambiados entre el usuario y el agente a lo largo de una misma sesión.

## 6.- Configurando el historial de conversaciones.     

Para configurar el historial de conversaciones es necesario importar los módulos `HumanMessage` y `AIMessage`del módulo de mensages principal de LangChain (`langchain.core_messages`).   
- Seguidamente, se configura una variable para almacenar todos los mensajes, comunmente denominada `message_history`.
- A continuación, definimos una nueva consulta, con una nueva pregunta sin información contextual adicional y con valores distintos para el cálculo.
- Se invoca al agente, ahora pasando tanto la nueva consulta como la variable que almacena todos los mensajes (historial de mensajes), todo ello dentro de un diccionario.   
- Se filtran únicamente los mensajes relevantes de la respuesta del agente (Usamos una "comprehension list" para seleccionar las instancias HumanMessage y AIMessage que disponen de contenido real). Al aplicar el metodo `strip()`se eliminan los espacios en blanco finales.   
- Finalmente, damos formato e imprimimos la conversación extraida del contenido de los mensajes, con cada mensaje etiquetado usando su nombre de clase correspondiente. Se observa que el `user_input`es ahora la nueva consulta mientras que el `agent_output`se corresponde con la consulta completa, cosa que resulta útil para depuración.

### 6.1 - Importamos nuevas clases necesarias.   

Importamos las clases `HumanMessage` y `AIMessage` para representar mensajes de entrada del usuario y respuestas del agente respectivamente

In [10]:
from langchain_core.messages import HumanMessage, AIMessage 

### 6.2 - Extracción del historial.   

Extraemos el historial de mensajes de la respuesta del agente

In [11]:
message_history = response["messages"]

### 6.3 - Definición de nuevas consultas.   

Se define una nueva consulta para el agente.

In [12]:
new_query = "¿Cuál es el área de un rectángulo con base 8 y altura 2?"

### 6.4 - Ejecución de la consulta.   

Lanzamos al agente pasando el historial de mensajes junto con la nueva consulta, lo que permite al agente mantener el contexto de la conversación y proporcionar una respuesta coherente basada en toda la información disponible.

In [13]:
messages = agent.invoke({"messages": message_history + [("user", new_query)]})

### 6.5 - Filtrado de mensajes.     

Se filtra el historial de mensajes para quedarnos solo con aquellos que son respuestas del agente (AIMessage) o entradas del usuario (HumanMessage), lo que nos permite centrarnos en la interacción relevante entre el usuario y el agente.

In [14]:
filtered_messages = [msg for msg in messages["messages"] 
                     if isinstance(msg, (HumanMessage, AIMessage))
                     and msg.content.strip()] 

### 6.6 - Elaboración de la salida (output).

Se construye un diccionario de salida que incluye la nueva consulta del usuario y una lista de las respuestas relevantes del agente, formateadas para mostrar claramente el tipo de mensaje (HumanMessage o AIMessage) junto con su contenido.

In [15]:
output ={
    "user_input": new_query,
    "agent_output": [f"{msg.__class__.__name__}: {msg.content}" for msg in filtered_messages]
}

Se formatea el output y se presenta el mensaje obtenido en markdown.

In [17]:
md = f"### 🧑 User Input\n{output['user_input']}\n\n### 🤖 Agent Output\n"
for line in output['agent_output']:
    md += f"{line}\n\n"
    
display(Markdown(md)) 

### 🧑 User Input
¿Cuál es el área de un rectángulo con base 8 y altura 2?

### 🤖 Agent Output
HumanMessage: ¿Cuál es la raíz cuadrada de 144 y puedes buscar noticias recientes sobre ese número y puedes decirme cúal es la capital de España?

AIMessage: 

La raíz cuadrada de 144 es **12**.  

En cuanto a noticias recientes sobre el número 144:  
- En WhatsApp, **144** puede significar que alguien está enfermo (1 = "I", 4 = "feel", 4 = "sick").  
- También se mencionó a un tenista que ganó 14 grandes torneos.  
- Hay referencias a noticias globales y eventos actuales.  

La capital de España es **Madrid**.  

¿Necesitas más detalles sobre alguno de estos temas?

HumanMessage: ¿Cuál es el área de un rectángulo con base 8 y altura 2?

AIMessage: 

El área de un rectángulo se calcula multiplicando la base por la altura.  

**Área = base × altura = 8 × 2 = 16**  

Por lo tanto, el área es **16 unidades cuadradas**.

